[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/087.%20The%20Hub-and-Spoke%20Network%20Design%20Problem/P87-Tier-7_executed.ipynb)

---


# 87. The Hub-and-Spoke Network Design Problem

## Tier 7: Human–AI Partnership

### Purpose and scope

This notebook illustrates a simulated human–AI collaboration around a small hub-and-spoke network-design example. It combines strategic inputs, a demand-weighted heuristic for selecting hub candidates, a cost evaluation for the resulting configuration, and simple measures of trust and acceptance across repeated interactions.

The network has 12 nodes. The model generates symmetric demand, uses Euclidean distances between node coordinates, and evaluates routes that travel from an origin to its assigned hub, between hubs, and from the destination hub to the destination. The parameter `alpha` discounts the inter-hub and final access legs in the route-cost calculation.

The hub-selection rule is a ranking heuristic: it scores nodes using demand, fixed hub cost, and a regional preference, then selects the highest-scoring candidates subject to the current maximum hub count. It is not an exact optimizer and does not guarantee a globally minimum-cost network. The cost calculation evaluates a selected configuration; it does not search all possible configurations.

The human partner is simulated by code. Strategic objectives, domain knowledge, evaluation, trust updates, and learning are illustrative inputs and rules, not measurements of real people or evidence that collaboration improves solution quality. Constraint values such as minimum service level and maximum cost inform some reported measures but are not enforced as feasibility constraints by the hub-selection heuristic.

### Modeling assumptions

- Node coordinates lie in a two-dimensional Euclidean plane. Distances are in coordinate units, not geographic miles or kilometers.
- Demand is generated randomly, made symmetric, and set to zero on the diagonal. No random seed is specified, so generated instances and simulated human inputs can differ between runs.
- Hub fixed costs, demand-weighted distances, and total cost are stylized quantities expressed in a common illustrative cost index. The model does not specify a real-world transportation rate or currency conversion.
- Each node is assigned to its nearest selected hub; a selected hub is assigned to itself. Capacity is reported through a utilization proxy but is not a selection constraint.
- The service-level, scalability, and risk scores are simplified indicators defined in the code. They should not be interpreted as calibrated operational performance measures.

### Collaboration sequence

Each iteration follows the same sequence: the simulated human provides a strategic objective and preferences; the AI-style heuristic ranks candidate hubs and evaluates a configuration; an explanation and alternatives are generated; the simulated human evaluates the recommendation; trust metrics are updated; and the interaction is recorded. Repeating this sequence makes the modeled feedback process visible, but does not establish that real trust or expertise would change in the same way.

### Mathematical model and heuristic

Let \(N=\{0,\ldots,n-1\}\) be the set of nodes, \(q_{ij}\) the demand from node \(i\) to node \(j\), \(d_{ij}\) the Euclidean distance, and \(H\subseteq N\) the selected hub set. Each node is assigned to its nearest selected hub. Write \(h(i)\in H\) for the assigned hub of node \(i\), with \(h(i)=i\) when \(i\in H\). The route distance used by the model for a pair \(i\ne j\) is

$$
L_{ij}(H)=d_{i,h(i)}+\alpha d_{h(i),h(j)}+\alpha d_{h(j),j}.
$$

The variable transportation component and total configuration cost are

$$
C_{\mathrm{var}}(H)=\sum_{i\in N}\sum_{\substack{j\in N\\j\ne i}}q_{ij}L_{ij}(H),
\qquad
C(H)=\sum_{h\in H}f_h+C_{\mathrm{var}}(H),
$$

where \(f_h\) is the fixed cost of opening hub \(h\). The sums include all ordered origin–destination pairs; symmetric demand means \(q_{ij}=q_{ji}\), but the two directions are still counted separately. In the code, `distance_matrix`, `demand_matrix`, `alpha`, `hub_costs`, and the `assignments` dictionary provide these quantities. The function `_evaluate_configuration` computes the assignment, route costs, and total cost.

The candidate-ranking score for node \(i\) is

$$
S_i=\left(\sum_{j\in N}q_{ij}+\sum_{j\in N}q_{ji}\right)
\frac{1}{1+f_i/50000}\,p_i,
$$

where \(p_i=1+\text{asia\_focus}\) for nodes whose first coordinate exceeds 60, and \(p_i=1\) otherwise. Nodes are sorted by decreasing \(S_i\), and the first `max_hubs` nodes are selected. This is the implemented decision rule; it is not equivalent to minimizing \(C(H)\) over all hub sets.

For each iteration, the trust update is also rule-based. If \(T_t\) is overall trust after iteration \(t\) and \(\Delta_t\) is the evaluation's trust impact, the code applies

$$
T_t=\min\left(0.95,\max\left(0.1,T_{t-1}+\Delta_t\right)\right).
$$

The initial value is 0.6. The evaluation rule supplies \(\Delta_t\); `_update_trust_metrics` applies the bounded update and records its history. Trust scores are model variables, not statistically estimated quantities.

### What to examine

- How strategic preferences affect candidate hub rankings and the resulting configuration.
- How fixed costs and demand-weighted route distances contribute to the evaluated cost.
- How the simulated acceptance rule feeds into the trust history.
- How the heuristic's scope differs from a constrained, globally optimal hub-location solution.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class StrategicObjective:
    """Represent a strategic objective supplied to the simulated AI."""

    name: str
    description: str
    priority: float
    constraints: List[str]


@dataclass
class DomainKnowledge:
    """Represent a simulated human knowledge area and its preferences."""

    area: str
    expertise_level: float
    insights: List[str]
    preferences: Dict[str, float]


@dataclass
class AIRecommendation:
    """Collect a configuration, its explanation, and related indicators."""

    solution: Dict
    confidence: float
    reasoning: List[str]
    alternatives: List[Dict]
    expected_outcomes: Dict[str, float]


@dataclass
class HumanEvaluation:
    """Represent the simulated human's evaluation of a recommendation."""

    acceptance_score: float
    modifications: Dict
    feedback: str
    trust_impact: float


@dataclass
class TrustMetrics:
    """Store bounded trust indicators used by the simulation."""

    competence: float
    integrity: float
    benevolence: float
    transparency: float
    reliability: float
    overall_trust: float


class HumanAISymbioticPartnership:
    """Simulate a human–AI process around hub-and-spoke design."""

    def __init__(self, num_nodes: int = 12):
        if num_nodes != 12:
            raise ValueError("This example provides coordinates for exactly 12 nodes.")

        self.num_nodes = num_nodes
        self.strategic_objectives = self._initialize_strategic_objectives()
        self.human_knowledge = self._initialize_human_knowledge()
        self.trust_metrics = self._initialize_trust_metrics()
        self.initial_trust = self.trust_metrics.overall_trust
        self._initialize_network_data()

        self.collaboration_history = []
        self.performance_history = []
        self.trust_history = [
            {
                "iteration": 0,
                "overall_trust": self.trust_metrics.overall_trust,
                "competence": self.trust_metrics.competence,
                "reliability": self.trust_metrics.reliability,
                "transparency": self.trust_metrics.transparency,
            }
        ]

    def _initialize_strategic_objectives(self) -> List[StrategicObjective]:
        """Create the strategic objectives used in the iteration cycle."""
        return [
            StrategicObjective(
                name="Cost Minimization",
                description="Reduce total network cost while maintaining service quality.",
                priority=0.30,
                constraints=["service_level >= 0.95", "max_hub_cost <= 50000"],
            ),
            StrategicObjective(
                name="Service Quality",
                description="Support service quality and reliability.",
                priority=0.25,
                constraints=["max_path_length <= 500", "hub_redundancy >= 2"],
            ),
            StrategicObjective(
                name="Scalability",
                description="Allow for future growth and expansion.",
                priority=0.20,
                constraints=["capacity_utilization <= 0.8", "expansion_nodes >= 3"],
            ),
            StrategicObjective(
                name="Risk Management",
                description="Consider operational risks and disruptions.",
                priority=0.15,
                constraints=["disruption_tolerance >= 0.9", "backup_routes >= 1"],
            ),
            StrategicObjective(
                name="Sustainability",
                description="Consider environmental sustainability.",
                priority=0.10,
                constraints=["carbon_efficiency >= 0.8", "green_hubs >= 0.3"],
            ),
        ]

    def _initialize_human_knowledge(self) -> List[DomainKnowledge]:
        """Create illustrative knowledge areas and regional preferences."""
        return [
            DomainKnowledge(
                area="Market Dynamics",
                expertise_level=0.90,
                insights=[
                    "Asian markets are described as growing.",
                    "European demand is described as stable but seasonal.",
                    "US markets are described as requiring faster delivery.",
                ],
                preferences={
                    "asia_priority": 0.8,
                    "europe_stability": 0.7,
                    "us_speed": 0.9,
                },
            ),
            DomainKnowledge(
                area="Infrastructure",
                expertise_level=0.85,
                insights=[
                    "Singapore is described as having strong connectivity.",
                    "German infrastructure is described as reliable.",
                    "US coastal hubs are described as efficient.",
                ],
                preferences={
                    "singapore_connectivity": 0.9,
                    "german_reliability": 0.95,
                    "us_coastal": 0.8,
                },
            ),
            DomainKnowledge(
                area="Regulatory Environment",
                expertise_level=0.80,
                insights=[
                    "EU regulations are described as favoring green logistics.",
                    "Asia is described as requiring local partnerships.",
                    "US regulations are described as flexible.",
                ],
                preferences={
                    "eu_green": 0.9,
                    "asia_local": 0.8,
                    "us_flexible": 0.7,
                },
            ),
        ]

    def _initialize_trust_metrics(self) -> TrustMetrics:
        """Set the initial illustrative trust values."""
        return TrustMetrics(
            competence=0.6,
            integrity=0.7,
            benevolence=0.5,
            transparency=0.6,
            reliability=0.6,
            overall_trust=0.6,
        )

    def _initialize_network_data(self) -> None:
        """Generate the example's coordinates, demands, costs, and capacities."""
        self.coordinates = np.array(
            [
                [0, 0],
                [40, 30],
                [80, 20],
                [60, 50],
                [20, 60],
                [90, 70],
                [30, 40],
                [70, 30],
                [50, 20],
                [10, 50],
                [85, 40],
                [25, 70],
            ],
            dtype=float,
        )

        self.demand_matrix = np.random.randint(
            50, 500, size=(self.num_nodes, self.num_nodes)
        )
        np.fill_diagonal(self.demand_matrix, 0)
        self.demand_matrix = (self.demand_matrix + self.demand_matrix.T) // 2

        coordinate_differences = (
            self.coordinates[:, np.newaxis, :] - self.coordinates[np.newaxis, :, :]
        )
        self.distance_matrix = np.sqrt(
            np.sum(coordinate_differences**2, axis=2)
        )

        self.hub_costs = np.random.randint(10000, 50000, size=self.num_nodes)
        self.hub_capacities = np.random.randint(1000, 5000, size=self.num_nodes)
        self.alpha = 0.75

    def collaborative_optimization(self, num_iterations: int = 10) -> Dict[str, float]:
        """Run the simulated strategic-input and evaluation cycle."""
        if num_iterations <= 0:
            raise ValueError("num_iterations must be a positive integer.")

        print("=== HUMAN–AI PARTNERSHIP SIMULATION ===")
        print(f"Initial trust score: {self.initial_trust:.2f}")

        for iteration in range(num_iterations):
            print(f"\n--- Iteration {iteration + 1} ---")

            strategic_input = self._get_human_strategic_input(iteration)
            ai_recommendation = self._ai_generate_solution(strategic_input)
            explanation = self._ai_explain_solution(ai_recommendation)
            human_evaluation = self._human_evaluate_recommendation(
                ai_recommendation, explanation
            )

            self._update_trust_metrics(human_evaluation)
            self._learn_from_collaboration(
                strategic_input, ai_recommendation, human_evaluation
            )
            self._record_collaboration(
                iteration,
                strategic_input,
                ai_recommendation,
                human_evaluation,
            )

            print(f"Trust score: {self.trust_metrics.overall_trust:.2f}")
            print(f"Heuristic confidence indicator: {ai_recommendation.confidence:.2f}")
            print(f"Simulated human acceptance: {human_evaluation.acceptance_score:.2f}")

        return self._generate_partnership_summary()

    def _get_human_strategic_input(self, iteration: int) -> Dict:
        """Assemble the objective, knowledge samples, and preferences."""
        return {
            "primary_objective": self.strategic_objectives[
                iteration % len(self.strategic_objectives)
            ],
            "domain_insights": random.sample(self.human_knowledge, 2),
            "specific_constraints": self._generate_adaptive_constraints(iteration),
            "preferences": self._generate_adaptive_preferences(iteration),
            "confidence_level": min(0.9, 0.6 + iteration * 0.03),
        }

    def _generate_adaptive_constraints(self, iteration: int) -> Dict[str, float]:
        """Supply illustrative limits that some reporting rules inspect."""
        constraints = {
            "max_hubs": 4,
            "min_service_level": 0.95,
            "max_cost": 200000,
        }

        if self.trust_metrics.overall_trust > 0.8:
            constraints["max_hubs"] = 5

        if iteration > 5:
            constraints["min_service_level"] = 0.97

        return constraints

    def _generate_adaptive_preferences(self, iteration: int) -> Dict[str, float]:
        """Represent the iteration-dependent preferences used in hub scoring."""
        return {
            "asia_focus": 0.8 if iteration > 3 else 0.6,
            "cost_sensitivity": 0.7 if iteration < 5 else 0.5,
            "quality_priority": 0.6 if iteration < 7 else 0.8,
            "risk_tolerance": 0.5 if iteration < 4 else 0.7,
        }

    def _ai_generate_solution(self, strategic_input: Dict) -> AIRecommendation:
        """Rank candidate hubs, evaluate the selected configuration, and report it."""
        max_hubs = int(
            strategic_input["specific_constraints"].get("max_hubs", 4)
        )
        if not 1 <= max_hubs <= self.num_nodes:
            raise ValueError("max_hubs must be between 1 and the number of nodes.")

        preferences = strategic_input["preferences"]
        hub_scores = []

        for node in range(self.num_nodes):
            demand_score = (
                np.sum(self.demand_matrix[node])
                + np.sum(self.demand_matrix[:, node])
            )
            cost_score = 1.0 / (1.0 + self.hub_costs[node] / 50000)
            preference_multiplier = 1.0

            if self.coordinates[node, 0] > 60:
                preference_multiplier *= (
                    1.0 + preferences.get("asia_focus", 0.0)
                )

            hub_scores.append(
                (node, demand_score * cost_score * preference_multiplier)
            )

        hub_scores.sort(key=lambda item: item[1], reverse=True)
        selected_hubs = [node for node, _ in hub_scores[:max_hubs]]
        solution = self._evaluate_configuration(selected_hubs)
        confidence = self._calculate_solution_confidence(
            solution, strategic_input
        )
        reasoning = self._generate_solution_reasoning(solution, strategic_input)
        alternatives = self._generate_alternative_solutions(
            solution, strategic_input
        )
        expected_outcomes = self._calculate_expected_outcomes(solution)

        return AIRecommendation(
            solution=solution,
            confidence=confidence,
            reasoning=reasoning,
            alternatives=alternatives,
            expected_outcomes=expected_outcomes,
        )

    def _evaluate_configuration(self, hub_locations: List[int]) -> Dict:
        """Assign nodes to nearest hubs and calculate stylized network measures."""
        if not hub_locations:
            raise ValueError("A configuration must contain at least one hub.")
        if len(set(hub_locations)) != len(hub_locations):
            raise ValueError("Hub locations must be unique.")
        if any(hub < 0 or hub >= self.num_nodes for hub in hub_locations):
            raise ValueError("Every hub index must identify a network node.")

        fixed_cost = float(sum(self.hub_costs[hub] for hub in hub_locations))
        assignments = {}

        for node in range(self.num_nodes):
            if node in hub_locations:
                assignments[node] = node
            else:
                assignments[node] = min(
                    hub_locations,
                    key=lambda hub: self.distance_matrix[node, hub],
                )

        variable_cost = 0.0
        total_weighted_path_length = 0.0
        total_flows = 0.0

        for origin in range(self.num_nodes):
            for destination in range(self.num_nodes):
                if origin == destination:
                    continue

                origin_hub = assignments[origin]
                destination_hub = assignments[destination]
                path_length = (
                    self.distance_matrix[origin, origin_hub]
                    + self.alpha
                    * self.distance_matrix[origin_hub, destination_hub]
                    + self.alpha
                    * self.distance_matrix[destination_hub, destination]
                )
                flow = self.demand_matrix[origin, destination]
                variable_cost += flow * path_length
                total_weighted_path_length += flow * path_length
                total_flows += flow

        hub_utilization = {}
        for hub in hub_locations:
            assigned_nodes = [
                node for node, assigned_hub in assignments.items()
                if assigned_hub == hub
            ]
            assigned_demand = sum(
                self.demand_matrix[node, other]
                + self.demand_matrix[other, node]
                for node in assigned_nodes
                for other in range(self.num_nodes)
                if node != other
            )
            hub_utilization[hub] = (
                assigned_demand / self.hub_capacities[hub]
            )

        average_path_length = (
            total_weighted_path_length / total_flows if total_flows > 0 else 0.0
        )
        total_cost = fixed_cost + variable_cost

        if not np.isclose(total_cost, fixed_cost + variable_cost):
            raise AssertionError("Total cost must equal its two components.")

        return {
            "hub_locations": list(hub_locations),
            "assignments": assignments,
            "total_cost": total_cost,
            "fixed_cost": fixed_cost,
            "variable_cost": variable_cost,
            "hub_utilization": hub_utilization,
            "avg_path_length": average_path_length,
        }

    def _calculate_solution_confidence(
        self, solution: Dict, strategic_input: Dict
    ) -> float:
        """Calculate a rule-based confidence indicator, not a probability."""
        base_confidence = 0.7
        alignment_bonus = 0.0

        objective_name = strategic_input["primary_objective"].name.lower()
        max_cost = strategic_input["specific_constraints"].get("max_cost", 250000)
        if "cost" in objective_name and solution["total_cost"] < max_cost:
            alignment_bonus += 0.1

        max_hubs = strategic_input["specific_constraints"].get("max_hubs", 4)
        if len(solution["hub_locations"]) <= max_hubs:
            alignment_bonus += 0.1

        trust_adjustment = self.trust_metrics.overall_trust * 0.1
        return min(0.95, base_confidence + alignment_bonus + trust_adjustment)

    def _generate_solution_reasoning(
        self, solution: Dict, strategic_input: Dict
    ) -> List[str]:
        """Explain the heuristic configuration and its measured cost."""
        max_cost = strategic_input["specific_constraints"].get("max_cost", 250000)
        budget_status = (
            "below" if solution["total_cost"] < max_cost else "at or above"
        )
        reasoning = [
            f"The ranking heuristic selected {len(solution['hub_locations'])} hubs.",
            f"The evaluated cost is {budget_status} the stated cost threshold.",
            "Candidate scores combine demand, fixed hub cost, and regional preference.",
        ]

        objective_name = strategic_input["primary_objective"].name
        if "Cost" in objective_name:
            reasoning.append(
                "Cost is the stated priority; the ranking rule itself is not a cost optimizer."
            )
        elif "Quality" in objective_name:
            reasoning.append(
                "Service-quality measures are reported as indicators, not enforced constraints."
            )
        elif "Scalability" in objective_name:
            reasoning.append(
                "Capacity utilization is reported as a proxy and does not restrict hub selection."
            )

        return reasoning

    def _generate_alternative_solutions(
        self, primary_solution: Dict, strategic_input: Dict
    ) -> List[Dict]:
        """Evaluate simple configuration variations for human consideration."""
        alternatives = []
        hubs = primary_solution["hub_locations"]

        if len(hubs) > 2:
            alternatives.append(self._evaluate_configuration(hubs[:-1]))

        if len(hubs) < 5:
            available_hubs = [
                node for node in range(self.num_nodes) if node not in hubs
            ]
            if available_hubs:
                added_hub = random.choice(available_hubs)
                alternatives.append(
                    self._evaluate_configuration(hubs + [added_hub])
                )

        return alternatives

    def _calculate_expected_outcomes(self, solution: Dict) -> Dict[str, float]:
        """Return illustrative indicators associated with a configuration."""
        return {
            "total_cost": solution["total_cost"],
            "service_level": self._calculate_service_level(solution),
            "scalability_score": self._calculate_scalability_score(solution),
            "risk_score": self._calculate_risk_score(solution),
            "implementation_complexity": (
                len(solution["hub_locations"]) / self.num_nodes
            ),
        }

    def _calculate_service_level(self, solution: Dict) -> float:
        """Map average route length to a bounded illustrative service indicator."""
        maximum_direct_distance = np.max(self.distance_matrix)
        if maximum_direct_distance == 0:
            return 0.99

        score = 1.0 - solution["avg_path_length"] / maximum_direct_distance
        return max(0.8, min(0.99, score))

    def _calculate_scalability_score(self, solution: Dict) -> float:
        """Use average hub utilization as a bounded scalability proxy."""
        utilizations = list(solution["hub_utilization"].values())
        average_utilization = float(np.mean(utilizations)) if utilizations else 0.0
        score = 1.0 - average_utilization
        return max(0.5, min(0.95, score))

    def _calculate_risk_score(self, solution: Dict) -> float:
        """Use hub count as a bounded, simplified risk proxy."""
        hub_count = len(solution["hub_locations"])
        score = 1.0 - hub_count / self.num_nodes
        return max(0.7, min(0.95, score))

    def _ai_explain_solution(self, recommendation: AIRecommendation) -> Dict:
        """Package cost components and heuristic reasoning for evaluation."""
        solution = recommendation.solution
        return {
            "summary": (
                f"The heuristic selected {len(solution['hub_locations'])} hubs."
            ),
            "key_factors": [
                f"Total cost index: {solution['total_cost']:,.0f}",
                f"Fixed-cost component: {solution['fixed_cost']:,.0f}",
                f"Variable-cost component: {solution['variable_cost']:,.0f}",
                f"Demand-weighted average route length: {solution['avg_path_length']:.1f}",
            ],
            "reasoning": recommendation.reasoning,
            "confidence_factors": [
                f"Rule-based confidence indicator: {recommendation.confidence:.2f}",
                f"Overall trust input: {self.trust_metrics.overall_trust:.2f}",
                f"Reliability indicator: {self.trust_metrics.reliability:.2f}",
            ],
            "alternatives_considered": len(recommendation.alternatives),
        }

    def _human_evaluate_recommendation(
        self, recommendation: AIRecommendation, explanation: Dict
    ) -> HumanEvaluation:
        """Simulate acceptance using confidence, trust, and explanation length."""
        base_acceptance = recommendation.confidence * 0.8
        trust_influence = self.trust_metrics.overall_trust * 0.2
        explanation_quality = len(explanation["key_factors"]) / 10.0
        acceptance_score = min(
            0.95, base_acceptance + trust_influence + explanation_quality
        )

        if acceptance_score > 0.8:
            feedback = "Strong simulated acceptance under the evaluation rule."
            trust_impact = 0.05
        elif acceptance_score > 0.6:
            feedback = "Moderate simulated acceptance under the evaluation rule."
            trust_impact = 0.02
        else:
            feedback = "Low simulated acceptance under the evaluation rule."
            trust_impact = -0.03

        modifications = {}
        if acceptance_score < 0.7:
            modifications["hub_adjustment"] = "Consider alternative hub locations."
            modifications["cost_reduction"] = "Review the cost components."

        return HumanEvaluation(
            acceptance_score=acceptance_score,
            modifications=modifications,
            feedback=feedback,
            trust_impact=trust_impact,
        )

    def _update_trust_metrics(self, evaluation: HumanEvaluation) -> None:
        """Apply the simulation's bounded trust-update rules."""
        self.trust_metrics.overall_trust = max(
            0.1,
            min(
                0.95,
                self.trust_metrics.overall_trust + evaluation.trust_impact,
            ),
        )

        if evaluation.acceptance_score > 0.8:
            self.trust_metrics.competence = min(
                0.95, self.trust_metrics.competence + 0.02
            )
            self.trust_metrics.reliability = min(
                0.95, self.trust_metrics.reliability + 0.02
            )

        if evaluation.trust_impact > 0:
            self.trust_metrics.transparency = min(
                0.95, self.trust_metrics.transparency + 0.01
            )
            self.trust_metrics.integrity = min(
                0.95, self.trust_metrics.integrity + 0.01
            )

        self.trust_history.append(
            {
                "iteration": len(self.trust_history),
                "overall_trust": self.trust_metrics.overall_trust,
                "competence": self.trust_metrics.competence,
                "reliability": self.trust_metrics.reliability,
                "transparency": self.trust_metrics.transparency,
            }
        )

    def _learn_from_collaboration(
        self,
        strategic_input: Dict,
        ai_recommendation: AIRecommendation,
        human_evaluation: HumanEvaluation,
    ) -> None:
        """Apply the example's small expertise update after high confidence."""
        if ai_recommendation.confidence > 0.8:
            for knowledge in self.human_knowledge:
                if knowledge.expertise_level < 0.9:
                    knowledge.expertise_level = min(
                        0.95, knowledge.expertise_level + 0.01
                    )

    def _record_collaboration(
        self,
        iteration: int,
        strategic_input: Dict,
        ai_recommendation: AIRecommendation,
        human_evaluation: HumanEvaluation,
    ) -> None:
        """Store iteration-level recommendation and evaluation indicators."""
        self.collaboration_history.append(
            {
                "iteration": iteration + 1,
                "strategic_objective": strategic_input["primary_objective"].name,
                "ai_confidence": ai_recommendation.confidence,
                "solution_cost": ai_recommendation.solution["total_cost"],
                "human_acceptance": human_evaluation.acceptance_score,
                "trust_score": self.trust_metrics.overall_trust,
                "feedback": human_evaluation.feedback,
            }
        )

        self.performance_history.append(
            {
                "iteration": iteration + 1,
                "solution_quality": ai_recommendation.confidence,
                "human_satisfaction": human_evaluation.acceptance_score,
                "partnership_effectiveness": (
                    ai_recommendation.confidence
                    + human_evaluation.acceptance_score
                )
                / 2,
            }
        )

    def _generate_partnership_summary(self) -> Dict[str, float]:
        """Summarize the recorded simulation history."""
        if not self.performance_history:
            raise ValueError("Run at least one collaboration iteration first.")

        return {
            "total_iterations": len(self.collaboration_history),
            "initial_trust": self.initial_trust,
            "final_trust": self.trust_metrics.overall_trust,
            "trust_improvement": (
                self.trust_metrics.overall_trust - self.initial_trust
            ),
            "avg_solution_quality": float(
                np.mean([item["solution_quality"] for item in self.performance_history])
            ),
            "avg_human_satisfaction": float(
                np.mean(
                    [item["human_satisfaction"] for item in self.performance_history]
                )
            ),
            "partnership_effectiveness": float(
                np.mean(
                    [
                        item["partnership_effectiveness"]
                        for item in self.performance_history
                    ]
                )
            ),
        }


print("Initializing the human–AI partnership simulation...")
partnership = HumanAISymbioticPartnership(num_nodes=12)

print("\nPartnership configuration:")
print(f"- Network size: {partnership.num_nodes} nodes")
print(f"- Strategic objectives: {len(partnership.strategic_objectives)}")
print(f"- Human knowledge areas: {len(partnership.human_knowledge)}")
print(f"- Initial trust score: {partnership.initial_trust:.2f}")

print("\nStrategic objectives:")
for objective in partnership.strategic_objectives:
    print(f"- {objective.name}: priority {objective.priority:.2f}")

print("\nHuman knowledge areas:")
for knowledge in partnership.human_knowledge:
    print(f"- {knowledge.area}: expertise {knowledge.expertise_level:.2f}")

assert partnership.demand_matrix.shape == (12, 12)
assert np.allclose(partnership.distance_matrix, partnership.distance_matrix.T)
assert np.all(np.diag(partnership.demand_matrix) == 0)
assert np.allclose(np.diag(partnership.distance_matrix), 0.0)

In [ ]:
partnership_summary = partnership.collaborative_optimization(num_iterations=10)

print("\n=== SIMULATION SUMMARY ===")
print(f"Total iterations: {partnership_summary['total_iterations']}")
print(f"Initial trust: {partnership_summary['initial_trust']:.2f}")
print(f"Final trust: {partnership_summary['final_trust']:.2f}")
print(f"Trust change: {partnership_summary['trust_improvement']:+.2f}")
print(
    "Average heuristic confidence indicator: "
    f"{partnership_summary['avg_solution_quality']:.2f}"
)
print(
    "Average simulated human acceptance: "
    f"{partnership_summary['avg_human_satisfaction']:.2f}"
)
print(
    "Average simulated partnership effectiveness: "
    f"{partnership_summary['partnership_effectiveness']:.2f}"
)

assert partnership_summary["total_iterations"] == 10
assert len(partnership.trust_history) == 11
assert np.isclose(
    partnership_summary["final_trust"] - partnership_summary["initial_trust"],
    partnership_summary["trust_improvement"],
)

In [ ]:
def visualize_human_ai_partnership(partnership):
    """Plot trust, simulated performance, and objective frequency."""
    if not partnership.performance_history:
        raise ValueError("Run the collaboration simulation before plotting.")

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(16, 12)
    )
    fig.suptitle("Human–AI Partnership Simulation", fontsize=16, fontweight="bold")

    trust_iterations = [item["iteration"] for item in partnership.trust_history]
    overall_trust = [item["overall_trust"] for item in partnership.trust_history]
    competence = [item["competence"] for item in partnership.trust_history]
    reliability = [item["reliability"] for item in partnership.trust_history]

    ax1.plot(trust_iterations, overall_trust, "o-", label="Overall trust")
    ax1.plot(trust_iterations, competence, "s-", label="Competence")
    ax1.plot(trust_iterations, reliability, "^-", label="Reliability")
    ax1.set_title("Trust Indicators by Iteration")
    ax1.set_xlabel("Iteration (0 is the initial value)")
    ax1.set_ylabel("Illustrative score")
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    performance = partnership.performance_history
    iterations = [item["iteration"] for item in performance]
    solution_quality = [item["solution_quality"] for item in performance]
    human_satisfaction = [item["human_satisfaction"] for item in performance]

    ax2.plot(iterations, solution_quality, "o-", label="Heuristic confidence indicator")
    ax2.plot(iterations, human_satisfaction, "s-", label="Simulated human acceptance")
    ax2.set_title("Rule-Based Evaluation Indicators")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Illustrative score")
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    effectiveness = [item["partnership_effectiveness"] for item in performance]
    ax3.bar(iterations, effectiveness, alpha=0.8, label="Per-iteration indicator")
    average_effectiveness = float(np.mean(effectiveness))
    ax3.axhline(
        y=average_effectiveness,
        color="red",
        linestyle="--",
        label=f"Mean: {average_effectiveness:.2f}",
    )
    ax3.set_title("Simulated Partnership Effectiveness")
    ax3.set_xlabel("Iteration")
    ax3.set_ylabel("Illustrative score")
    ax3.set_ylim(0, 1)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    objective_counts = {}
    for record in partnership.collaboration_history:
        objective = record["strategic_objective"]
        objective_counts[objective] = objective_counts.get(objective, 0) + 1

    objectives = list(objective_counts)
    counts = [objective_counts[objective] for objective in objectives]
    bars = ax4.bar(objectives, counts, alpha=0.8)
    ax4.set_title("Strategic Objective by Iteration")
    ax4.set_xlabel("Strategic objective")
    ax4.set_ylabel("Number of iterations")
    ax4.tick_params(axis="x", rotation=35)
    ax4.grid(True, axis="y", alpha=0.3)

    for bar, count in zip(bars, counts):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(count),
            ha="center",
            va="bottom",
        )

    fig.tight_layout()
    plt.show()
    return fig


partnership_figure = visualize_human_ai_partnership(partnership)

## Interpreting the simulation

The charts summarize the rules implemented in the notebook, not observed human behavior or evidence of improved optimization performance. The trust plot includes the initial trust value at iteration 0, followed by the recorded values after each simulated evaluation. The objective-frequency plot reflects the objective sequence used by the iteration cycle.

A higher trust or acceptance indicator does not establish that the selected network is better. The hub configuration comes from a demand-, cost-, and preference-based ranking heuristic; the network cost is evaluated afterward. To assess a particular run, inspect the recommendation's hub locations, assignments, cost components, average route length, utilization indicators, and the constraints that are not enforced by the heuristic.

### Strengths and limitations of this partnership model

The simulation makes strategic inputs, explanations, evaluation, and feedback explicit. This separation helps readers see where human-provided preferences enter the heuristic and how evaluation rules affect the recorded trust indicators. The alternative configurations also provide concrete candidates for discussion.

The example is deliberately stylized. Its simulated evaluation rules do not represent a validated model of human judgment; its confidence indicator is not a probability; and its service, scalability, and risk measures are simplified proxies. The heuristic does not enforce all listed strategic constraints, model disruption or backup routes, or prove optimality. Real applications would require domain-validated data, operationally meaningful units, explicit feasibility checks, and human evaluation grounded in the relevant organization and decision context.

### When this type of workflow may be useful

A human–AI workflow can be useful when domain knowledge and strategic priorities must be made visible alongside computational analysis. It is especially important to distinguish recommendations from constraints, indicators from validated outcomes, and a heuristic candidate from a proven optimum. In this example, the central learning goal is to trace those distinctions through the data, ranking rule, cost calculation, simulated feedback, and plots.
